# Volatility Clustering, EWMA Volatility

09/01 ended with returns that are uncorrelated while their squares are strongly autocorrelated: turbulent days follow turbulent
days. A single volatility number for the whole sample then describes no actual day well. This notebook measures volatility
that changes over time, first with a rolling window and then with the exponentially weighted moving average (EWMA) of RiskMetrics,
and uses it for value at risk and volatility targeting. 09/03 replaces EWMA with a model estimated by maximum likelihood.

## 0. Math link

There is no volatility-modelling notebook in the course repos yet (curriculum gap). This notebook builds on:

- [Limits of sequences](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/01_limits/00_limits_of_sequences.ipynb):
  the EWMA weights $(1-\lambda)\lambda^{k-1}$ form a geometric series that sums to 1.
- [Variance](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/05_variance.ipynb):
  a volatility estimate is a weighted average of squared returns.
- Autocorrelation of squared returns from [09/01](./01_random_walk_and_autocorrelation.ipynb); value at risk and its backtest from
  [05/01](../05_risk_measures/01_value_at_risk.ipynb).

## 1. Introduction

A risk manager computes daily VaR from one year of returns. After a calm year, the VaR is low exactly when a crisis starts; after a
crisis, it stays high for a year after markets have calmed down. Weighting recent returns more heavily fixes much of this.

After this notebook you can:

- describe volatility clustering and why it produces fat tails in unconditional returns;
- compute rolling-window and EWMA volatility and explain the ghost effect;
- derive the EWMA weights, their half-life and effective window;
- choose the decay factor $\lambda$ by forecast accuracy;
- build an EWMA value at risk and compare its exceedances with a constant-volatility VaR;
- scale a position to a volatility target.

## 2. Theory

### Volatility clustering

Let $r_t = \sigma_t z_t$ with i.i.d. $z_t$ of mean 0 and variance 1, and a conditional volatility $\sigma_t$ that depends on the past. Then $E[r_t \mid \text{past}] = 0$ (no
autocorrelation), but $E[r_t^2 \mid \text{past}] = \sigma_t^2$ varies. Mixing calm and turbulent days gives unconditional returns with fatter tails than $z_t$:
with normal $z_t$, the kurtosis of $r_t$ is $3E[\sigma^4]/E[\sigma^2]^2 > 3$ whenever $\sigma_t$ varies.

### Rolling window

$$
\hat\sigma^2_t = \frac1M\sum_{k=1}^M r_{t-k}^2
$$

(daily mean set to zero, standard for daily risk). Every return in the window has weight $1/M$, then drops out. A large return
raises the estimate for exactly $M$ days and then disappears abruptly: the **ghost effect**.

### EWMA

RiskMetrics (1996) uses the recursion

$$
\hat\sigma^2_t = \lambda\hat\sigma^2_{t-1} + (1-\lambda)r_{t-1}^2, \qquad 0 < \lambda < 1
$$

Unrolling it:

$$
\hat\sigma^2_t = (1-\lambda)\sum_{k=1}^\infty \lambda^{k-1}r_{t-k}^2
$$

The weights $(1-\lambda)\lambda^{k-1}$ decline geometrically and sum to $(1-\lambda)/(1-\lambda) = 1$. Useful summaries:

- **half-life** $h = \ln 0.5/\ln\lambda$: the lag at which the weight has halved;
- **mean lag** $\sum_k k(1-\lambda)\lambda^{k-1} = 1/(1-\lambda)$;
- weight beyond lag $K$: $\lambda^K$.

RiskMetrics uses $\lambda = 0.94$ for daily data (half-life 11.2 days, mean lag 16.7 days) and 0.97 for monthly.

### Choosing $\lambda$

$\hat\sigma^2_t$ is a forecast of $r_t^2$, made before $r_t$ is known. Choose $\lambda$ to minimize a loss over the sample, for example

$$
\text{MSE} = \frac1n\sum_t (r_t^2 - \hat\sigma^2_t)^2 \qquad \text{or} \qquad \text{QLIKE} = \frac1n\sum_t\Big(\ln\hat\sigma^2_t + \frac{r_t^2}{\hat\sigma^2_t}\Big)
$$

QLIKE is the negative Gaussian log-likelihood up to constants; it is less dominated by the few largest squared returns.

### Forecasts and horizons

EWMA has no long-run level: the forecast for every future day is today's $\hat\sigma^2$, so the $h$-day variance forecast is $h\hat\sigma^2$. (It is the
special case of GARCH(1,1) with $\omega = 0$ and $\alpha + \beta = 1$; 09/03 adds mean reversion.)

### Using it

- **VaR**: $\mathrm{VaR}_t = z_\alpha\hat\sigma_t V$. With a good volatility estimate, exceedances should be about $1 - \alpha$ of days **and** not cluster.
- **Standardized returns** $r_t/\hat\sigma_t$ are closer to i.i.d. and have thinner tails than $r_t$.
- **Volatility targeting**: hold exposure $\sigma^*/\hat\sigma_t$ to keep risk near a target $\sigma^*$.
- **Covariances**: the same recursion on cross-products $r_{i,t}r_{j,t}$ gives an EWMA covariance matrix (exercise 5).

## 3. Worked Examples

### Example 1: one update

Yesterday's EWMA volatility 1.0% (daily), yesterday's return −3%, $\lambda = 0.94$:

$$
\hat\sigma^2 = 0.94\cdot 0.0001 + 0.06\cdot 0.0009 = 0.000094 + 0.000054 = 0.000148, \qquad \hat\sigma = 1.22\%
$$

One 3-sigma day raises the volatility estimate by 22%.

### Example 2: half-lives

$\lambda = 0.94$: $h = \ln 0.5/\ln 0.94 = 11.2$ days. $\lambda = 0.97$: 22.8 days. $\lambda = 0.99$: 69 days.

### Example 3: weights

With $\lambda = 0.94$: yesterday 6%, 20 days ago $0.06\cdot 0.94^{19} = 1.85\%$, all returns older than 75 days together $0.94^{75} = 1\%$.

### Example 4: ghost effect

A 60-day window and a −8% day: the variance estimate rises by $0.0064/60 = 0.000107$ for 60 days. If daily volatility was 1% (variance 0.0001), the
estimate jumps to $\sqrt{0.000207} = 1.44\%$, stays there (other things equal) and falls back to 1% on day 61, when nothing happened.

### Example 5: volatility targeting

Target 10% annual, current EWMA volatility 25% annual: exposure $10/25 = 40\%$ of capital. If volatility falls to 8%: $10/8 = 125\%$ (leverage).

## 4. Finance Application

### Where it matters

- **Value at risk.** Bank and asset-manager VaR systems use EWMA or GARCH volatility (and EWMA covariances) so that VaR reacts to market
  conditions; regulators require backtests that also look at clustering of exceedances.
- **Margin and collateral.** Clearing houses scale initial margins with recent volatility, with floors to avoid procyclicality.
- **Volatility targeting and risk parity.** Position sizes scale inversely with EWMA volatility (08/03 levered risk parity typically targets
  a volatility this way).
- **Performance analysis.** Volatility-standardized returns compare strategies across calm and turbulent periods.

### The usual misreadings

- "Annualized EWMA volatility is the volatility for the next year." EWMA's forecast is flat; it says nothing about mean reversion over the year.
- "The VaR doubled, so the portfolio is twice as risky." The portfolio may be unchanged; the estimate of the market's volatility moved.
  Whether that is right depends on how persistent the shock is.
- "Higher $\lambda$ is more accurate because it uses more data." It reacts more slowly; the best $\lambda$ trades noise against delay.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
trading_days = 252

### Simulated returns with clustered volatility

Twenty years of daily returns from a GARCH(1,1) process with Student-$t$ innovations (5 degrees of freedom, scaled to variance 1). Long-run
volatility 16% per year. Because the data are simulated, the true conditional volatility $\sigma_t$ is known and estimators can be checked
against it. (09/03 explains and estimates the model.)

In [ ]:
n = 20 * trading_days
alpha_g, beta_g, nu = 0.08, 0.90, 5
var_long = (0.16 / np.sqrt(trading_days)) ** 2
omega = var_long * (1 - alpha_g - beta_g)

z = rng.standard_t(nu, n) / np.sqrt(nu / (nu - 2))
h = np.empty(n)
r = np.empty(n)
h[0] = var_long
for t in range(n):
    if t > 0:
        h[t] = omega + alpha_g * r[t - 1] ** 2 + beta_g * h[t - 1]
    r[t] = np.sqrt(h[t]) * z[t]
sigma_true = np.sqrt(h)

print(f"annualized volatility: {r.std() * np.sqrt(trading_days):.2%}")
print(f"true conditional vol (annualized): min {sigma_true.min() * np.sqrt(252):.1%}, max {sigma_true.max() * np.sqrt(252):.1%}")
print(f"kurtosis: returns {stats.kurtosis(r, fisher=False):.2f}, innovations z {stats.kurtosis(z, fisher=False):.2f}")

The true conditional volatility ranges from 8% to 46% (annualized). The returns have kurtosis 9.0, more than the 7.3 of the $t_5$ innovations
alone: clustering adds to the fat tails.

### Rolling and EWMA estimators

In [ ]:
def rolling_vol(r, M):
    out = np.full(len(r), np.nan)
    c = np.concatenate([[0.0], np.cumsum(r**2)])
    for t in range(M, len(r)):
        out[t] = np.sqrt((c[t] - c[t - M]) / M)          # uses r_{t-M} ... r_{t-1}
    return out


def ewma_vol(r, lam, init=None):
    s2 = np.empty(len(r))
    s2[0] = np.mean(r[:20] ** 2) if init is None else init
    for t in range(1, len(r)):
        s2[t] = lam * s2[t - 1] + (1 - lam) * r[t - 1] ** 2
    return np.sqrt(s2)


est = {f"rolling {M}d": rolling_vol(r, M) for M in [21, 63, 252]}
est.update({f"EWMA {lam}": ewma_vol(r, lam) for lam in [0.94, 0.97]})
start = 252                                               # compare where all estimators are defined
rmse = {k: np.sqrt(np.mean((v[start:] - sigma_true[start:]) ** 2)) * np.sqrt(252) for k, v in est.items()}
pd.Series(rmse).map(lambda v: f"{v:.2%}").rename("RMSE vs true vol (annualized)")

EWMA with $\lambda = 0.94$ tracks the true volatility best (RMSE 1.5 points), followed by a 21-day window (2.3). Long windows lag badly: the 252-day
estimate misses by 5.1 points on average.

### Choosing $\lambda$

In [ ]:
lams = np.round(np.arange(0.80, 0.996, 0.005), 3)
mse, qlike = [], []
for lam in lams:
    s2 = ewma_vol(r, lam)[start:] ** 2
    mse.append(np.mean((r[start:] ** 2 - s2) ** 2))
    qlike.append(np.mean(np.log(s2) + r[start:] ** 2 / s2))
lam_mse, lam_ql = lams[np.argmin(mse)], lams[np.argmin(qlike)]
rmse_true = [np.sqrt(np.mean((ewma_vol(r, l)[start:] - sigma_true[start:]) ** 2)) for l in lams]
print(f"lambda minimizing MSE: {lam_mse}, QLIKE: {lam_ql}, RMSE vs true vol: {lams[np.argmin(rmse_true)]}")
for l in [0.94, 0.97]:
    print(f"half-life for lambda {l}: {np.log(0.5) / np.log(l):.1f} days")

QLIKE picks 0.94, the RiskMetrics value; MSE picks 0.935; the $\lambda$ that tracks the true volatility best is 0.92. For this process all three are
close.

### Standardized returns

In [ ]:
s_ewma = ewma_vol(r, 0.94)
std_rows = {
    "raw returns / full-sample vol": r[start:] / r.std(),
    "returns / EWMA 0.94 vol": r[start:] / s_ewma[start:],
    "returns / true vol": r[start:] / sigma_true[start:],
}
pd.DataFrame({k: {"std": v.std(), "kurtosis": stats.kurtosis(v, fisher=False),
                  "rho_1 of squares": np.corrcoef(v[1:] ** 2, v[:-1] ** 2)[0, 1]} for k, v in std_rows.items()}).T.round(3)

Dividing by the EWMA volatility removes the autocorrelation of squared returns (0.10 → 0.01) and lowers the kurtosis from 9.4 to 8.2. Even
with the true volatility the kurtosis stays at 7.5: that part comes from the $t_5$ innovations, not from clustering.

### Value at risk: constant vs EWMA

99% one-day VaR for a CHF 1,000,000 position, normal quantile. Constant: volatility of the previous 252 days. EWMA: $\lambda = 0.94$.

In [ ]:
V = 1_000_000
z99 = stats.norm.ppf(0.99)
var_const = z99 * rolling_vol(r, 252) * V
var_ewma = z99 * s_ewma * V
loss = -r * V

def backtest(var_series, label):
    ok = ~np.isnan(var_series)
    ok[:start] = False
    exc = loss[ok] > var_series[ok]
    p_after = exc[1:][exc[:-1]].mean() if exc[:-1].sum() else np.nan
    per_year = exc.reshape(-1, 252).sum(axis=1) if exc.size % 252 == 0 else None
    return {"VaR": label, "exceedance rate": exc.mean(), "expected": 0.01,
            "P(exceedance | exceedance yesterday)": p_after,
            "max exceedances in a year": per_year.max() if per_year is not None else np.nan,
            "min exceedances in a year": per_year.min() if per_year is not None else np.nan}

pd.DataFrame([backtest(var_const, "252-day constant"), backtest(var_ewma, "EWMA 0.94")]).set_index("VaR").round(4)

Both VaRs are exceeded more often than 1% of days (2.2% and 1.9%), because the normal quantile is too small for the fat-tailed innovations
(a $t$ quantile or historical simulation of standardized returns would fix the level). The difference is in the clustering: after an exceedance,
the constant VaR is exceeded again the next day with probability 4.7%, the EWMA VaR with 1.1%. The worst year has 13 exceedances for the
constant VaR and 9 for EWMA.

### Volatility targeting

Scale exposure daily to a 10% annual target using yesterday's EWMA volatility (capped at 200%).

In [ ]:
target = 0.10 / np.sqrt(252)
expo = np.minimum(target / s_ewma, 2.0)
r_vt = expo * r
yearly = pd.DataFrame({"unscaled": r[start:].reshape(-1, 252).std(axis=1) * np.sqrt(252),
                       "vol-targeted": r_vt[start:].reshape(-1, 252).std(axis=1) * np.sqrt(252)})
print(f"realized annual vol across years: unscaled {yearly['unscaled'].min():.1%} to {yearly['unscaled'].max():.1%}, "
      f"vol-targeted {yearly['vol-targeted'].min():.1%} to {yearly['vol-targeted'].max():.1%}")
print(f"kurtosis: unscaled {stats.kurtosis(r[start:], fisher=False):.2f}, vol-targeted {stats.kurtosis(r_vt[start:], fisher=False):.2f}")

Unscaled, the realized volatility ranges from 9.7% to 27.6% across years; vol-targeted, from 9.8% to 11.5%. The fat tails remain (kurtosis
8.2): targeting works on clustering, not on the jumps within a day.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
years = np.arange(n) / 252
ax.plot(years, r, color="grey", linewidth=0.4, label="daily return")
ax.plot(years, 2.33 * sigma_true, color="tab:red", linewidth=0.8, label="±2.33 × true conditional vol")
ax.plot(years, -2.33 * sigma_true, color="tab:red", linewidth=0.8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Simulated daily returns with volatility clustering")
ax.set_xlabel("years")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
win = slice(int(np.argmax(sigma_true)) - 250, int(np.argmax(sigma_true)) + 250)
d = np.arange(n)[win]
ax.plot(d, sigma_true[win] * np.sqrt(252), color="black", linewidth=1.5, label="true")
ax.plot(d, est["rolling 63d"][win] * np.sqrt(252), label="rolling 63 days")
ax.plot(d, est["rolling 252d"][win] * np.sqrt(252), label="rolling 252 days")
ax.plot(d, est["EWMA 0.94"][win] * np.sqrt(252), label="EWMA 0.94")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Volatility estimates around the most turbulent period (annualized)")
ax.set_xlabel("day")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
k = np.arange(1, 121)
for lam in [0.94, 0.97]:
    axes[0].plot(k, (1 - lam) * lam ** (k - 1), label=f"EWMA {lam}")
axes[0].plot(k, np.where(k <= 63, 1 / 63, 0), label="rolling 63 days")
axes[0].set_title("Weight on the squared return k days ago")
axes[0].set_xlabel("lag k (days)")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
axes[0].legend()

axes[1].plot(lams, np.array(rmse_true) * np.sqrt(252), color="tab:blue", label="RMSE vs true vol (annualized)")
ax2 = axes[1].twinx()
ax2.plot(lams, qlike, color="tab:orange", label="QLIKE")
axes[1].axvline(lam_ql, color="tab:orange", linestyle=":")
axes[1].set_title("Choosing lambda")
axes[1].set_xlabel("lambda")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
axes[1].legend(loc="upper left")
ax2.legend(loc="upper right")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ok = np.arange(n) >= start
for lab, vs, c in [("252-day constant", var_const, "tab:blue"), ("EWMA 0.94", var_ewma, "tab:orange")]:
    exc_days = np.flatnonzero(ok & (loss > vs))
    axes[0].plot(exc_days / 252, np.full(exc_days.size, 1 if lab.startswith("252") else 0), "|", markersize=14, color=c, label=lab)
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(["EWMA 0.94", "252-day"])
axes[0].set_ylim(-0.5, 1.5)
axes[0].set_title("Days with a loss above the 99% VaR")
axes[0].set_xlabel("years")
x_pos = np.arange(len(yearly))
axes[1].bar(x_pos - 0.2, yearly["unscaled"], width=0.4, label="unscaled")
axes[1].bar(x_pos + 0.2, yearly["vol-targeted"], width=0.4, label="vol-targeted (10%)")
axes[1].axhline(0.10, color="black", linewidth=0.8)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].set_xticks(x_pos[::2])
axes[1].set_xticklabels(x_pos[::2] + 2)
axes[1].set_title("Realized volatility by year")
axes[1].set_xlabel("year (the first year is used to start the estimates)")
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Yesterday's EWMA volatility was 1.5% (daily) and yesterday's return +4%. Compute today's estimate with $\lambda = 0.94$ and with $\lambda = 0.97$.
2. For $\lambda = 0.97$, compute the half-life, the mean lag and the total weight on returns older than 100 days.
3. A risk report shows that the portfolio's EWMA VaR doubled overnight after a −5% market day, without any trades. **Interpret** this for the
   risk committee: is the portfolio riskier, and how quickly will the VaR come down if markets calm?

### Intermediate

4. Plot MSE and QLIKE against $\lambda$ for the simulated series. Why does MSE prefer a different $\lambda$ than QLIKE, and which one is closer to the
   $\lambda$ that best tracks the true volatility?
5. EWMA correlation: simulate two assets with daily volatility 1% whose correlation switches from 0.2 to 0.8 on day 1,000 (2,000 days in total). Track
   the EWMA correlation ($\lambda = 0.94$ for variances and covariance) and the rolling 250-day correlation. How long until each estimate is above 0.7?

### Advanced

6. Show that EWMA is a GARCH(1,1) with $\omega = 0$ and $\alpha + \beta = 1$, and that its $h$-day variance forecast is $h\hat\sigma^2_{t+1}$. Compare the 20-day volatility forecast of
   EWMA with the true GARCH forecast (mean-reverting) on the day of the highest true volatility in the sample.
7. Ghost effect: add a single −10% return to an otherwise i.i.d. 1% daily series and plot the rolling 63-day and EWMA 0.94 volatility for 100 days.

## 8. Solutions / Checks

In [ ]:
# 1.
for lam in [0.94, 0.97]:
    print(f"1. lambda {lam}: {np.sqrt(lam * 0.015**2 + (1 - lam) * 0.04**2):.3%}")

# 2.
lam = 0.97
print(f"2. half-life {np.log(0.5) / np.log(lam):.1f} days, mean lag {1 / (1 - lam):.1f} days, weight beyond 100 days {lam**100:.1%}")

# 4.
print(f"4. MSE prefers {lam_mse}, QLIKE {lam_ql}, true-vol RMSE {lams[np.argmin(rmse_true)]}")

In [ ]:
# 5.
n5 = 2_000
rho_path = np.where(np.arange(n5) < 1_000, 0.2, 0.8)
e1, e2 = rng.standard_normal(n5), rng.standard_normal(n5)
a5 = 0.01 * e1
b5 = 0.01 * (rho_path * e1 + np.sqrt(1 - rho_path**2) * e2)
s11, s22, s12 = [np.empty(n5) for _ in range(3)]
s11[0], s22[0], s12[0] = 1e-4, 1e-4, 0.2e-4
for t in range(1, n5):
    s11[t] = 0.94 * s11[t - 1] + 0.06 * a5[t - 1] ** 2
    s22[t] = 0.94 * s22[t - 1] + 0.06 * b5[t - 1] ** 2
    s12[t] = 0.94 * s12[t - 1] + 0.06 * a5[t - 1] * b5[t - 1]
corr_ewma = s12 / np.sqrt(s11 * s22)
corr_roll = np.array([np.corrcoef(a5[t - 250:t], b5[t - 250:t])[0, 1] if t >= 250 else np.nan for t in range(n5)])
def first_above(x, lvl=0.7, start_=1_000):
    idx = np.flatnonzero(x[start_:] > lvl)
    return idx[0] if idx.size else None
print(f"5. days after the switch until corr > 0.7: EWMA {first_above(corr_ewma)}, rolling 250 {first_above(corr_roll)}")
print(f"   noise before the switch (std of estimate, days 300-999): EWMA {corr_ewma[300:1000].std():.3f}, rolling {corr_roll[300:1000].std():.3f}")

# 6.
t_peak = int(np.argmax(sigma_true))
h_next = omega + alpha_g * r[t_peak] ** 2 + beta_g * h[t_peak]
garch_20 = sum(var_long + (alpha_g + beta_g) ** (j) * (h_next - var_long) for j in range(20))
ewma_next = 0.94 * s_ewma[t_peak] ** 2 + 0.06 * r[t_peak] ** 2
print(f"6. 20-day vol forecast at the peak: EWMA {np.sqrt(20 * ewma_next):.2%}, true GARCH {np.sqrt(garch_20):.2%}")

In [ ]:
# 7.
r7 = 0.01 * rng.standard_normal(400)
r7[200] = -0.10
fig, ax = plt.subplots(figsize=(9, 4))
d7 = np.arange(150, 300)
ax.plot(d7, rolling_vol(r7, 63)[d7] * np.sqrt(252), label="rolling 63 days")
ax.plot(d7, ewma_vol(r7, 0.94)[d7] * np.sqrt(252), label="EWMA 0.94")
ax.axvline(200, color="grey", linestyle=":")
ax.axvline(263, color="grey", linestyle=":")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("A single -10% day: ghost effect of the rolling window")
ax.set_xlabel("day")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

**1.** $\lambda = 0.94$: $\sqrt{0.94\cdot 0.000225 + 0.06\cdot 0.0016} = 1.75\%$. $\lambda = 0.97$: $\sqrt{0.97\cdot 0.000225 + 0.03\cdot 0.0016} = 1.63\%$.

**2.** Half-life 22.8 days, mean lag 33.3 days, weight beyond 100 days $0.97^{100} = 4.8\%$.

**3.** The portfolio's positions are unchanged; the model's estimate of market volatility jumped, because a −5% day has a large weight (6%)
in the EWMA. If the shock reflects a lasting change in conditions, the higher VaR is right; if not, the estimate decays by the factor 0.94 per day
in variance, with a half-life of about 11 days. The committee should look at the size of the move relative to recent volatility and
at other indicators (implied volatility) rather than react mechanically.

**4.** MSE is dominated by the few largest squared returns, which are very noisy measures of variance; QLIKE weights relative errors and is
closer to the likelihood. In this simulation both pick values near the $\lambda$ that tracks the true volatility best; QLIKE is the standard choice.

**5.** The EWMA correlation first exceeds 0.7 after 43 days, the rolling 250-day estimate after 193 days. Before the switch the EWMA estimate is
much noisier (standard deviation 0.15 around the true 0.2, against 0.03 for the rolling estimate). Speed and noise trade off exactly as for volatility.

**6.** With $\omega = 0$ and $\alpha + \beta = 1$ the GARCH recursion is the EWMA recursion with $\lambda = \beta$. Its forecasts $E[\sigma^2_{t+j}]$ stay at $\hat\sigma^2_{t+1}$ for every $j$ because the
mean-reversion factor $\alpha + \beta = 1$. At the volatility peak the EWMA 20-day forecast is higher than the true GARCH forecast, which already
expects volatility to fall back toward its long-run level.

**7.** The rolling estimate jumps on day 201 and drops abruptly 63 days later, when the −10% day leaves the window; the EWMA jumps and decays smoothly.

## 9. Conclusion

- Volatility clusters: returns are uncorrelated, their squares are not, and mixing calm and turbulent periods fattens the tails.
- A rolling window weights every return equally and produces ghost effects; EWMA weights them by $(1-\lambda)\lambda^{k-1}$, with half-life $\ln 0.5/\ln\lambda$.
- $\lambda$ can be chosen by minimizing a forecast loss; QLIKE is the likelihood-based choice.
- EWMA VaR reacts to market conditions and gives fewer clustered exceedances than a one-year constant VaR; volatility targeting keeps realized
  risk near a target.
- EWMA forecasts are flat over the horizon; GARCH (09/03) adds mean reversion toward a long-run level.

Next: [09/03 GARCH(1,1) by maximum likelihood](./03_garch_1_1_by_maximum_likelihood.ipynb).